In [ ]:
!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.23.5 --force-reinstall # numpyのバージョンを修正し、強制的に再インストール
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    IN_COLAB = False
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
# 必要なモジュールをインポート
import sys
from pathlib import Path
from IPython.display import HTML
from base64 import b64encode

# プロジェクトのルートディレクトリをパスに追加
sys.path.insert(0, '.')

# プロジェクトモジュールをインポート
from src.pipelines.player_pose_exporter import PlayerPoseExporter

print("✓ モジュールのインポートが完了しました")

In [ ]:
INPUT_VIDEO = 'data/raw/sample_video_01_short.MOV'
OUTPUT_VIDEO = 'output/player_classification_result.mp4'
CSV_OUTPUT = 'output/player_pose_data.csv' 

In [ ]:
# モデルパス
TABLE_MODEL_PATH = 'models/table_detection/best.pt'
POSE_MODEL_PATH = 'models/pretrained/yolo11l-pose.pt'

# 処理パラメータ
FPS = 30.0              # 処理FPS（GPU利用時は高FPS推奨）
MAX_PLAYERS = 4         # 最大プレイヤー数
MIN_PLAYER_SCORE = 0.3  # プレイヤー判定の最小スコア閾値（0.0-1.0）
MIN_CONSECUTIVE_FRAMES = 30  # CSV出力時に保持する最小連続フレーム数
MAX_FRAME_GAP = 5           # 連続性を判定する際の最大フレーム間隔

print("テストパラメータ:")
print(f"  入力動画: {INPUT_VIDEO}")
print(f"  出力動画: {OUTPUT_VIDEO}")
print(f"  CSV出力: {CSV_OUTPUT}")
print(f"  処理FPS: {FPS}")
print(f"  最大プレイヤー数: {MAX_PLAYERS}")
print(f"  最小スコア閾値: {MIN_PLAYER_SCORE}")
print(f"  最小連続フレーム数: {MIN_CONSECUTIVE_FRAMES}")
print(f"  最大フレーム間隔: {MAX_FRAME_GAP}")

In [ ]:
# パイプラインを初期化
pipeline = PlayerPoseExporter(
    table_model_path=TABLE_MODEL_PATH,
    pose_model_path=POSE_MODEL_PATH,
    device='cuda',
    max_players=MAX_PLAYERS,
    min_player_score=MIN_PLAYER_SCORE
)

# 動画を処理
results = pipeline.process_video(
    input_video=INPUT_VIDEO,
    output_video=OUTPUT_VIDEO,
    csv_output=CSV_OUTPUT,
    target_fps=FPS,
    min_consecutive_frames=MIN_CONSECUTIVE_FRAMES,
    max_frame_gap=MAX_FRAME_GAP,
    show_progress=True
)

print(f"\n処理結果:")
print(f"  総フレーム数: {results['total_frames']}")
print(f"  処理フレーム数: {results['processed_frames']}")
print(f"  プレイヤーID: {results['player_ids']}")
print(f"  候補者数: {results['candidates_count']}")
